# U-Net like CNN without skip-connections (TRAIN)

## Imports
### Libs

In [ ]:
from pathlib import Path

import torch
from clearml import Dataset, Task
from torch import nn
from torch.utils.data import DataLoader

### Chromatica modules

In [ ]:
from chromatica.datasets.dataset import ImageDataset
from chromatica.nn.v1.cnn import CNN

### Check for CUDA or MPS

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is used")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is used")
else:
    device = torch.device("cpu")
    print("CPU is used")

## Train task init

In [ ]:
task = Task.init(
    project_name="Chromatica",
    task_name="Train U-Net like without skip-connections",
)

In [ ]:
params = {
    "learning_rate": 1e-4,
    "num_epochs": 10,
    "batch_size": 32,
    "num_workers": 6,
}
params = task.connect_configuration(params)

## Load dataset

In [ ]:
path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="Food101").get_local_copy()
)

In [ ]:
dataset = ImageDataset(path / "train")

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=["batch_size"],
    shuffle=True,
    num_workers=params["num_workers"],
    persistent_workers=True,
    pin_memory=(device == torch.device("cuda")),
)

## Train

In [ ]:
model = CNN().to(device)

In [ ]:
%%time

criterion = nn.MSELoss()
optim = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

model.train()
for epoch in range(params["num_epochs"]):
    epoch_loss = 0
    for x_, y_, _ in loader:
        x = x_.to(device, non_blocking=True)
        y = y_.to(device, non_blocking=True)

        optim.zero_grad()

        pred = model(x)
        loss = criterion(pred, y)

        loss.backward()
        optim.step()
        epoch_loss += loss.item()

    task.get_logger().report_scalar("Loss", "train", epoch_loss, epoch)

In [ ]:
model_path = Path("./.data")
model_path.mkdir(exist_ok=True, parents=True)
torch.save(model.state_dict(), model_path / "nn_v1.pth")
task.upload_artifact("model", model_path / "nn_v1.pth")

In [ ]:
task.mark_completed()